# 🎯 Bonus Stage — Streamlit Deployment Notebook

This notebook demonstrates the **complete pipeline** behind our interactive Vietnamese Extractive News Summarization app.

It mirrors exactly what `app.py` does so you can:
- Understand each step
- Test the logic on any input
- Verify that the Streamlit app works correctly before deployment

### What we built
An interactive web app where users can:
1. Paste any Vietnamese news article
2. Choose a summarization method (Lead-k, Vanilla LexRank, Position-Aware LexRank, or PA-LexRank+MMR)
3. Adjust parameters via sliders
4. See the summary, redundancy score, and which sentences were selected
5. Download the summary as a `.txt` file
6. Compare all 4 methods side by side

---

## Step 0: Install dependencies

In [ ]:
# Run this once to install dependencies
# %pip install streamlit scikit-learn numpy

## Step 1: Import libraries

In [ ]:
import numpy as np
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

print('✅ Libraries loaded')

## Step 2: Load a test article

In [ ]:
# Load from test_samples or paste directly
with open('test_samples/sample_economic.txt', encoding='utf-8') as f:
    raw = f.read()

# Extract title and body
lines = raw.strip().split('\n')
TITLE = lines[0].replace('Tiêu đề: ', '').strip()
BODY = '\n'.join(l for l in lines[2:] if not l.startswith('---'))

print(f'Title: {TITLE}')
print(f'Body length: {len(BODY)} characters')

## Step 3: Sentence splitting

In [ ]:
def sentence_split(text: str) -> list:
    """Split Vietnamese text into sentences using regex."""
    text = text.strip()
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return sentences

sentences = sentence_split(BODY)
print(f'Total sentences: {len(sentences)}')
for i, s in enumerate(sentences):
    print(f'  [{i}] {s[:80]}...' if len(s) > 80 else f'  [{i}] {s}')

## Step 4: TF-IDF similarity matrix

In [ ]:
def build_similarity_matrix(sentences: list, threshold: float = 0.1) -> np.ndarray:
    """Compute cosine similarity matrix using character n-gram TF-IDF.
    Character n-grams work well for Vietnamese without word segmentation."""
    if len(sentences) < 2:
        return np.eye(len(sentences))
    vectorizer = TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        min_df=1,
        sublinear_tf=True
    )
    tfidf = vectorizer.fit_transform(sentences)
    sim_matrix = cosine_similarity(tfidf, tfidf)
    sim_matrix[sim_matrix < threshold] = 0.0  # threshold pruning
    np.fill_diagonal(sim_matrix, 0.0)
    return sim_matrix

sim_matrix = build_similarity_matrix(sentences, threshold=0.1)
print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'Non-zero edges: {np.count_nonzero(sim_matrix)}')
print('\nTop similarity pairs:')
pairs = [(sim_matrix[i][j], i, j) for i in range(len(sentences)) for j in range(i+1, len(sentences))]
for score, i, j in sorted(pairs, reverse=True)[:5]:
    print(f'  [{i}]↔[{j}]: {score:.4f}')

## Step 5: LexRank — Power Iteration

In [ ]:
def lexrank_scores(sim_matrix: np.ndarray, max_iter: int = 100, damping: float = 0.85) -> np.ndarray:
    """Compute LexRank centrality via power iteration (like PageRank on sentences)."""
    n = len(sim_matrix)
    row_sums = sim_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    norm_matrix = sim_matrix / row_sums
    
    scores = np.ones(n) / n
    for iteration in range(max_iter):
        new_scores = (1 - damping) / n + damping * norm_matrix.T @ scores
        if np.linalg.norm(new_scores - scores) < 1e-6:
            print(f'Converged at iteration {iteration}')
            break
        scores = new_scores
    return scores

lr_scores = lexrank_scores(sim_matrix)
print('LexRank scores (per sentence):')
for i, score in enumerate(lr_scores):
    print(f'  [{i}] {score:.5f}  — {sentences[i][:60]}...')

## Step 6: Position prior + Title similarity

In [ ]:
def position_prior(num_sentences: int, position_weight: float = 0.8) -> np.ndarray:
    """Exponential decay: earlier sentences receive higher prior."""
    positions = np.arange(num_sentences)
    prior = np.exp(-positions * position_weight / num_sentences)
    prior /= prior.sum()
    return prior

def title_similarity_scores(sentences: list, title: str) -> np.ndarray:
    """Score each sentence by similarity to the title."""
    if not title.strip():
        return np.ones(len(sentences)) / len(sentences)
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)
    corpus = [title] + sentences
    tfidf = vectorizer.fit_transform(corpus)
    scores = cosine_similarity(tfidf[0], tfidf[1:]).flatten()
    scores = scores / scores.sum() if scores.sum() > 0 else np.ones(len(sentences)) / len(sentences)
    return scores

pos_prior = position_prior(len(sentences), position_weight=0.8)
title_scores = title_similarity_scores(sentences, TITLE)

# Combine: 70% LexRank + 30% title, then blend with 70% combined + 30% position
combined = lr_scores * 0.7 + title_scores * 0.3
combined = combined * 0.7 + pos_prior * 0.3
combined /= combined.sum()

print('Combined (LexRank + Position + Title) scores:')
ranked = sorted(enumerate(combined), key=lambda x: x[1], reverse=True)
for rank, (i, score) in enumerate(ranked):
    print(f'  Rank {rank+1}: [{i}] score={score:.5f}  — {sentences[i][:60]}...')

## Step 7: MMR Selection

In [ ]:
def mmr_select(sentences: list, scores: np.ndarray, k: int, lambda_mmr: float = 0.7) -> list:
    """
    Maximal Marginal Relevance:
      MMR(i) = lambda * relevance(i) - (1-lambda) * max_similarity_to_selected(i)
    """
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)
    tfidf = vectorizer.fit_transform(sentences)
    full_sim = cosine_similarity(tfidf, tfidf)
    
    remaining = list(range(len(sentences)))
    selected = []
    
    for step in range(k):
        if not remaining:
            break
        if not selected:
            best = max(remaining, key=lambda i: scores[i])
        else:
            best, best_score = None, -1e9
            for i in remaining:
                rel = scores[i]
                red = max(full_sim[i][j] for j in selected)
                mmr_score = lambda_mmr * rel - (1 - lambda_mmr) * red
                if mmr_score > best_score:
                    best_score, best = mmr_score, i
        print(f'  Step {step+1}: selected sentence [{best}] → {sentences[best][:70]}...')
        selected.append(best)
        remaining.remove(best)
    
    return sorted(selected)

K = 5
print(f'MMR selection (k={K}, λ=0.7):')
selected_indices = mmr_select(sentences, combined, k=K, lambda_mmr=0.7)
print(f'\nSelected indices: {selected_indices}')

## Step 8: Generate and evaluate the summary

In [ ]:
def compute_redundancy(sentences: list, indices: list) -> float:
    """Average pairwise cosine similarity between selected sentences."""
    if len(indices) < 2:
        return 0.0
    selected_sents = [sentences[i] for i in indices]
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)
    tfidf = vectorizer.fit_transform(selected_sents)
    sim = cosine_similarity(tfidf, tfidf)
    n = len(selected_sents)
    pairs = [(i, j) for i in range(n) for j in range(i+1, n)]
    return float(np.mean([sim[i][j] for i, j in pairs]))

summary = ' '.join(sentences[i] for i in selected_indices)
redundancy = compute_redundancy(sentences, selected_indices)
compression = 1 - len(selected_indices) / len(sentences)

print('=' * 80)
print('📝 SUMMARY (Position-Aware LexRank + MMR)')
print('=' * 80)
print(summary)
print()
print(f'Sentences selected:  {len(selected_indices)} / {len(sentences)}')
print(f'Compression ratio:   {compression:.1%}')
print(f'Redundancy:          {redundancy:.4f}  (lower = better)')

## Step 9: Compare all 4 methods

In [ ]:
def lead_k(sentences, k):
    return list(range(min(k, len(sentences))))

def vanilla_lexrank(sentences, k, threshold=0.1):
    sim_mat = build_similarity_matrix(sentences, threshold)
    scores = lexrank_scores(sim_mat)
    return sorted(np.argsort(scores)[-k:].tolist())

def position_lexrank(sentences, k, title='', threshold=0.1, position_weight=0.8):
    sim_mat = build_similarity_matrix(sentences, threshold)
    lr = lexrank_scores(sim_mat)
    pos = position_prior(len(sentences), position_weight)
    ts = title_similarity_scores(sentences, title)
    sc = lr * 0.7 + ts * 0.3
    sc = sc * 0.7 + pos * 0.3
    sc /= sc.sum()
    return sorted(np.argsort(sc)[-k:].tolist())

def position_lexrank_mmr(sentences, k, title='', threshold=0.1, position_weight=0.8, lambda_mmr=0.7):
    sim_mat = build_similarity_matrix(sentences, threshold)
    lr = lexrank_scores(sim_mat)
    pos = position_prior(len(sentences), position_weight)
    ts = title_similarity_scores(sentences, title)
    sc = lr * 0.7 + ts * 0.3
    sc = sc * 0.7 + pos * 0.3
    sc /= sc.sum()
    return mmr_select(sentences, sc, k, lambda_mmr)

methods = {
    'Lead-k': lead_k(sentences, K),
    'Vanilla LexRank': vanilla_lexrank(sentences, K),
    'Position-Aware LexRank': position_lexrank(sentences, K, TITLE),
    'Position-Aware LexRank + MMR': position_lexrank_mmr(sentences, K, TITLE),
}

print(f'{'Method':<35} {'Indices':<25} {'Redundancy':>12}')
print('-' * 75)
for name, sel in methods.items():
    red = compute_redundancy(sentences, sel)
    print(f'{name:<35} {str(sel):<25} {red:>12.4f}')

## Step 10: Launch the Streamlit app

In [ ]:
# Run this cell to launch the Streamlit app locally
# It will open automatically in your browser at http://localhost:8501

import subprocess, sys
print('Launching Streamlit app...')
print('→ Open http://localhost:8501 in your browser')
print('→ Press Ctrl+C or stop the kernel to quit')

# Uncomment to run:
# subprocess.run([sys.executable, '-m', 'streamlit', 'run', 'app.py'])

## Step 11: Deploy to Streamlit Community Cloud

```
1. Push the bonus_submission/ folder to GitHub (public repo)
2. Go to https://share.streamlit.io
3. Sign in with GitHub → New app
4. Repository: your-username/your-repo
5. Main file path: app.py
6. Click Deploy → live in ~2 minutes
```

Your app will be accessible at:
`https://your-username-your-repo-app-xxxxx.streamlit.app`

---

## Summary for Report

| Aspect | Detail |
|--------|--------|
| **App type** | Streamlit web app |
| **NLP task** | Vietnamese extractive news summarization |
| **Model** | Position-Aware LexRank + MMR |
| **Input** | Any Vietnamese news article + optional title |
| **Output** | Extractive summary, redundancy score, sentence breakdown |
| **Deployment** | Streamlit Community Cloud (free, no server needed) |
| **Unique features** | 4-method comparison, adjustable sliders, download button |
| **No dependencies** | Works with only scikit-learn + numpy (no spaCy, underthesea, etc.) |